**Fluorine-18 decay and PET scan timing.**
Fluorine-18 (${}^{18}\text{F}$) is a positron-emitting radioisotope with a half-life
of $t_{1/2} = 1.833$ hours.
It is attached to glucose molecules to form **FDG** (fluorodeoxyglucose),
which is injected into patients before a **PET scan** (Positron Emission Tomography).
Cancer cells absorb FDG at a higher rate than normal tissue, making them
visible as bright spots in the scan.

Because ${}^{18}\text{F}$ decays rapidly, the scan must be completed while enough
radioactive signal remains.
As derived in the Carbon-14 notebook, the decay equation is:

$$\frac{dn}{dt} = -\frac{\ln 2}{t_{1/2}}\,n
\qquad\Rightarrow\qquad
n(t) = n_0\,2^{-t/t_{1/2}}$$

This notebook uses Euler's method to simulate the decay,
then uses the analytic solution to compute the hard time limit
for a useful PET scan.

---
**Simulation parameters.**
The simulation runs for 12 hours (about 6.5 half-lives) with 100 steps
of $\Delta t = 0.12$ hours (7.2 minutes each).

In [ ]:
"""fluorine18_decay.ipynb"""

# Cell 01 - Simulation parameters

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

tf = 12  # final time (hours)
ts = 100  # number of time steps
dt = tf / ts  # time step size (0.12 hours = 7.2 minutes)

print(f"{tf=:,}  {ts=:,}  {dt=:.2f}")

**Pre-allocating the time and concentration arrays.**

In [ ]:
# Cell 02 - Allocate arrays for time (hours) and nuclei concentration (%)

t = np.zeros(ts)
n = np.zeros(ts)

pd.DataFrame({"t": t[:5], "n": n[:5]})

**Initial conditions.**
The half-life of ${}^{18}\text{F}$ is $t_{1/2} = 1.833$ hours (109.97 minutes).
The simulation starts at 100% concentration.

In [ ]:
# Cell 03 - Initial conditions: 100% concentration, F-18 half-life

n[0] = 100  # initial concentration (100%)
tau = 1.833  # half-life of F-18 (hours)

pd.DataFrame({"t": t[:5], "n": n[:5]})

**Euler forward integration.**
The decay constant $\lambda = \ln 2 / t_{1/2}$ converts the half-life
into the correct rate for the ODE $dn/dt = -\lambda\,n$.

In [ ]:
# Cell 04 - Euler forward integration of the F-18 decay equation

decay_const = np.log(2) / tau  # lambda = ln(2) / half-life

for i in range(ts - 1):
    t[i + 1] = t[i] + dt
    n[i + 1] = n[i] - decay_const * n[i] * dt

pd.DataFrame({"t": t[:5], "n": n[:5]})

**Decay curve and PET scan time limit.**
As of 2026, the best PET detector technology can detect ${}^{18}\text{F}$
concentrations down to approximately **5%** of the original dose.
Below this level the signal-to-noise ratio is too poor to produce
a diagnostic image.

Solving the analytic solution for the time at which 5% remains:

$$0.05 = 2^{-t_{\max}/t_{1/2}}
\qquad\Rightarrow\qquad
t_{\max} = -t_{1/2}\,\log_2(0.05)
= \frac{t_{1/2}\,\ln 20}{\ln 2}$$

This gives the hard outer limit for any useful scan.
In clinical practice, scans are performed well before this limit
(typically 60-90 minutes post-injection) to ensure sufficient image quality.

In [ ]:
# Cell 05 - Plot decay curve and compute maximum scan window

n_min_pct = 5.0  # detection limit (%)
t_max = -tau * np.log(n_min_pct / 100) / np.log(2)  # hours

print(f"Detection limit      : {n_min_pct}% of original F-18")
print(f"Half-lives elapsed   : {t_max / tau:.2f}")
print(f"Maximum scan window  : {t_max:.2f} hours ({t_max * 60:.0f} minutes)")

t_exact = np.linspace(0, tf, 500)
n_exact = 100 * 2 ** (-t_exact / tau)

plt.figure("fluorine18_decay", figsize=(9, 5))
plt.plot(t, n, label="Euler's method")
plt.plot(t_exact, n_exact, "--", label=r"Analytic $n_0\,2^{-t/\tau}$")
plt.axhline(
    n_min_pct, color="red", linestyle=":", label=f"Detection limit ({n_min_pct}%)"
)
plt.axvline(
    t_max,
    color="orange",
    linestyle=":",
    label=f"Max scan window ({t_max:.2f} hr / {t_max * 60:.0f} min)",
)
plt.scatter([t_max], [n_min_pct], color="red", zorder=5)
plt.title("Fluorine-18 Decay - PET Scan Timing")
plt.xlabel("Time (hours)")
plt.ylabel("% Concentration")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()